# Data Analysis (2): Pandas 2

 
## Contents 
In this `data_analysis_2` notebook we use more advanced features of the `pandas` package:
1. Merging data sets using `pandas`
2. Adding fields to dataframes 
3. Combining fields in dataframes 
4. Deleting fields from dataframes

In [2]:
# Imports ...
import numpy as np
import pandas as pd

/Users/matthewjohnpayne/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


# 1. Merging Datasets 

In many “real world” situations, the data that we want to use come in multiple files. 

We often need to combine these files into a single DataFrame to analyze the data. 

The pandas package provides various methods for combining DataFrames including `merge` and `concat`.

In the example below we will create some simple data, then import them into dataframes, and then **merge** the data using the `merge` function. 

## 1.1 Create Sample Data 

Here we create two dataframes directly from dictionaries using the approach demonstrated in [fundamentals_14.ipynb](../A_fundamentals/fundamentals_14.ipynb)

Note that some of the values in the `Name` field are the same in the two dataframes, but some are different...

In [35]:
# --- DATAFRAME 1: AGES ----
# Create a dictionary and then use this to create a Pandas DataFrame
df_ages = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie", "David"],
    "Age": [25, 30, 28, 22]
    })


# --- DATAFRAME 2: STATES ----
# Create a dictionary and then use this to create a Pandas DataFrame
df_states = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie", "Elaine"],
    "state": ["MA", "TX", "CA", "MA"]
    })



In [36]:
df_ages.head()

,Name,Age
0,Alice,25
1,Bob,30
2,Charlie,28
3,David,22


In [37]:
df_states.head()

,Name,state
0,Alice,MA
1,Bob,TX
2,Charlie,CA
3,Elaine,MA


## 1.2 Merge the dataframes

We can tell the `merge` function which field name to use as a basis for assessing similarity:
 - Use the the `on` option to tell `pandas` to use the "Names" field

N.B. The default method used to `merge` only keeps the values that are present in **both** fields 

In [41]:
df_merge = df_ages.merge(df_states, on='Name')
df_merge.head()

,Name,Age,state
0,Alice,25,MA
1,Bob,30,TX
2,Charlie,28,CA


If we want to keep **all** of the entries, then we have to use the optional `how` field. 
 - See [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) for more description of all the options.
 
In the example below we use the "outer" option and that keeps all data:
 - It forces "NaN" values to appear in some cases

In [42]:
df_merge = df_ages.merge(df_states, on='Name', how='outer')
df_merge.head()

,Name,Age,state
0,Alice,25.0,MA
1,Bob,30.0,TX
2,Charlie,28.0,CA
3,David,22.0,NaN
4,Elaine,NaN,MA


Note that in the above `df_merge` dataframe, the rows that were only present in one of the initial dataframes have `NaN` values in some of the rows. 

# Exercise Pandas 2(a) : Merging Data Sets

You have been provided with two csv files. 
The first file, "mass_midm.dat", contains a list of stars and their masses that were calculated based on ... 
The second file, "mass_reddot.dat", contains a list of stars and their masses that were calculated based on ...

We want to understand which stars have had their mass measured using *both* methods. 

You should ...
1. Read the two sets of data into two dataframes.
 - E.g. "mass_midm.dat" into `df_m` & "mass_reddot.dat" into `df_r`
 
 
2. You probably need to "tidy-up" the data in the `df_m` & `df_r` dataframes. 
 - (a) If you look at the column names in the dataframe (using `df_r.columns` & `df_m.columns`), you will probably see that there are lots of spaces in the names:
   - Get rid of these using a command like `df_r.columns = df_r.columns.str.replace(' ', '')`
   
   
3. Merge together the two dataframes by matching them on the `PreferredName` field
 - How many stars are present in *both* dataframes?
 - How many stars are present in *either* one or both of the dataframes (use `how='outer'` in the merge function)

In [191]:
len(df_r), len(df_m), len(df_merge) 

(1457, 413, 1612)

# 1.3 Add New Field to Dataframe

Consider the stock data below.

It contains both the total stock of each vegetable and the price-per-item for each vegetable

Let's see how we can add in a field that contains the total value of each item ...


In [204]:
# Create the dataframe ...
df_veg = pd.DataFrame({
    "Name": ["Cabbage", "Lettuce", "Carrot", "Cucumber"],
    "Stock": [15, 40, 38, 32],
    "Unit Price": [1.50, 1.75, 0.09, 1.25]
    })
df_veg.head()

,Name,Stock,Unit Price
0,Cabbage,15,1.50
1,Lettuce,40,1.75
2,Carrot,38,0.09
3,Cucumber,32,1.25


In [206]:
# Add an empty field to the dataframe 
df_veg['Total Value'] = None
df_veg.head()

,Name,Stock,Unit Price,Total Value
0,Cabbage,15,1.50,None
1,Lettuce,40,1.75,None
2,Carrot,38,0.09,None
3,Cucumber,32,1.25,None


In [207]:
# Use the data from the `Stock` & `Unit Price` fields to populate the `Total Value` field 
df_veg['Total Value'] = df_veg['Stock'] * df_veg['Unit Price']
df_veg.head()

,Name,Stock,Unit Price,Total Value
0,Cabbage,15,1.50,22.50
1,Lettuce,40,1.75,70.00
2,Carrot,38,0.09,3.42
3,Cucumber,32,1.25,40.00


In [208]:
# N.B. There is no need to set up the field separately ... you can do everything in one step 
df_veg['Another Total Value Field'] = df_veg['Stock'] * df_veg['Unit Price']
df_veg.head()

,Name,Stock,Unit Price,Total Value,Another Total Value Field
0,Cabbage,15,1.50,22.50,22.50
1,Lettuce,40,1.75,70.00,70.00
2,Carrot,38,0.09,3.42,3.42
3,Cucumber,32,1.25,40.00,40.00


# 1.4 Coalesce Fields 

We may want to combine data sets and take information from both wherever it is available 

Consider the stock data below.
 - One dataframe has stock data for vegetables 
 - The other dataframe has stock data for fruit
 
Let's combine them together into an overall produce dataframe
 - When we do that we see that it produces separate stock columns for each input dataframe (`Stock_x` & `Stock_y`)

In [209]:
# --- DATAFRAME 1: VEGETABLES ----
# Create a dictionary and then use this to create a Pandas DataFrame
df_v = pd.DataFrame({
    "Name": ["Cabbage", "Lettuce", "Carrot", "Cucumber"],
    "Stock": [15, 40, 38, 32]
    })


# --- DATAFRAME 2: FRUIT ----
# Create a dictionary and then use this to create a Pandas DataFrame
df_f = pd.DataFrame({
    "Name": ["Banana", "Strawberry", "Orange"],
    "Stock": ["22", "33", "11"]
    })

df_produce = df_v.merge(df_f, on='Name', how='outer')
df_produce.head(7)

,Name,Stock_x,Stock_y
0,Cabbage,15.0,NaN
1,Lettuce,40.0,NaN
2,Carrot,38.0,NaN
3,Cucumber,32.0,NaN
4,Banana,NaN,22
5,Strawberry,NaN,33
6,Orange,NaN,11


Let's create a `Combined Stock` field and populate it using the `combine_first` function

In [210]:
df_produce['Combined Stock'] = df_produce.Stock_x.combine_first(df_produce.Stock_y)
df_produce.head(7)

,Name,Stock_x,Stock_y,Combined Stock
0,Cabbage,15.0,NaN,15.0
1,Lettuce,40.0,NaN,40.0
2,Carrot,38.0,NaN,38.0
3,Cucumber,32.0,NaN,32.0
4,Banana,NaN,22,22
5,Strawberry,NaN,33,33
6,Orange,NaN,11,11


# 1.4 Drop Unwanted Fields

The original fields may not be useful any more
If so, we can `drop` the fields ...

In [211]:
df_produce.drop(['Stock_x', 'Stock_y'], axis=1, inplace=True)
df_produce

,Name,Combined Stock
0,Cabbage,15.0
1,Lettuce,40.0
2,Carrot,38.0
3,Cucumber,32.0
4,Banana,22
5,Strawberry,33
6,Orange,11


# Exercise Pandas 2(b) : Combining Data Sets

Take the merged dataframe from Exercise 2(a) that was produced using the `how='outer'` method to preserve all stars in the dataset

1. Add a field `mass (db-first)` and populate it using the mass from the `mass-db` field if that is a number, and then otherwise populate it with the mass from the `mass-r` field 
2. Add a field `mass (r-first)` and populate it using the mass from the `mass-r` field if that is a number, and then otherwise populate it with the mass from the `mass-db` field 
3. Drop the original `mass-db` and `mass-r` fields


In [214]:
df_merge.drop(['mass-db', 'mass-r'], axis=1, inplace=True)
df_merge

,PreferredName,e_mass,mass (db-first),mass (r-first)
0,GJ1001-B,NaN,0.040,0.040
1,GJ1001-C,NaN,0.040,0.040
2,GJ1001-A,0.015,0.234,0.262
3,GJ0001,NaN,0.411,0.411
4,LHS1019,NaN,0.335,0.335
...,...,...,...,...
1607,LEHPM1-5031,0.014,0.111,0.111
1608,LP876-010-C,0.014,0.196,0.196
1609,LEP2302+4338,0.014,0.184,0.184
1610,LEHPM2-2163,0.014,0.138,0.138
